# Ejercicio 9: Uso de la API de Google Gemini

### **Estudiante:** Kevin Alvear

En este ejercicio vamos a aprender a utilizar la API de OpenAI

## 1. Uso básico

El siguiente código sirve para conectarse con la API de Google Gemini de forma básica

In [23]:
import os
from google import genai

# Leer API key desde archivo gapi.txt (debe estar en el mismo directorio)
try:
    with open("gapi.txt", "r") as f:
        api_key = f.read().strip()
except FileNotFoundError:
    raise FileNotFoundError("No se encontró el archivo gapi.txt. Asegúrate de que esté en el directorio actual.")

# Crear cliente
client = genai.Client(api_key=api_key)
print("Conectado a Gemini (google-genai)")

Conectado a Gemini (google-genai)


## 2. Retrieval

### 2.1 Cargo el corpus de 20 News Groups

In [24]:
from sklearn.datasets import fetch_20newsgroups

# Seleccionar algunas categorías para reducir el tamaño
categories = ['alt.atheism', 'soc.religion.christian', 'comp.graphics', 'sci.med']
newsgroups = fetch_20newsgroups(subset='all', categories=categories, shuffle=True, random_state=42)

# Textos y etiquetas
documents = newsgroups.data
targets = newsgroups.target
target_names = newsgroups.target_names

print(f"Documentos cargados: {len(documents)}")
print(f"Categorías: {target_names}")

Documentos cargados: 3759
Categorías: ['alt.atheism', 'comp.graphics', 'sci.med', 'soc.religion.christian']


In [25]:
import re

def clean_newsgroup(text):
    """
    Limpia textos del dataset 20 Newsgroups eliminando:
    - Cabeceras (headers) hasta la primera línea en blanco.
    - Firmas (todo lo que sigue a '-- ').
    - Líneas citadas (que empiezan con '>').
    - URLs y correos electrónicos (opcional, se reemplazan por marcadores).
    - Espacios en blanco y caracteres no ASCII excesivos.
    """
    # 1. Separar el cuerpo del mensaje (descartar cabeceras)
    lines = text.split('\n')
    body_start = 0
    for i, line in enumerate(lines):
        if line.strip() == '':
            body_start = i + 1
            break
    text = '\n'.join(lines[body_start:])

    # 2. Eliminar firmas (común en estos grupos)
    text = re.sub(r'-- .*$', '', text, flags=re.MULTILINE)

    # 3. Eliminar líneas de citas (respuestas anidadas)
    text = re.sub(r'^>.*$', '', text, flags=re.MULTILINE)

    # 4. Normalizar espacios y saltos de línea
    text = re.sub(r'\n\s*\n', '\n\n', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # 5. Eliminar caracteres no imprimibles (mantener ASCII básico)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)

    # 6. (Opcional) Ofuscar correos y URLs para evitar ruido
    text = re.sub(r'\S+@\S+', '[EMAIL]', text)
    text = re.sub(r'http\S+', '[URL]', text)

    return text

print("Aplicando limpieza a los documentos...")
documents = [clean_newsgroup(doc) for doc in documents]
print(f"Limpieza completada. Total de documentos: {len(documents)}")
print("\n--- Ejemplo de texto limpio ---")
print(documents[0][:500] + "...")

Aplicando limpieza a los documentos...
Limpieza completada. Total de documentos: 3759

--- Ejemplo de texto limpio ---
In article [EMAIL] [EMAIL] (Paul Schmidt) writes: Where did you read this? I don't think this is true. I think most medical treatments are based on science, although it is difficult to prove anything with certitude. It is true that there are some things that have just been found "to work", but we have no good explanation for why. But almost everything does have a scientific rationale. The most common treatment for prostate cancer is probably hormone therapy. It has been "proven" to work. So have...


### 2.2 Transformo a embeddings

In [26]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Modelo de embeddings ligero y rápido
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generar embeddings para todos los documentos (en lotes)
print("Generando embeddings...")
embeddings = embedding_model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)
print(f"Embeddings generados: {embeddings.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6273.15it/s]


Generando embeddings...


Batches: 100%|██████████| 118/118 [03:14<00:00,  1.65s/it]

Embeddings generados: (3759, 384)


### 2.3 Creo una query y hago la búsqueda

In [27]:
import faiss
import time

# Construir índice FAISS (producto interno = coseno con vectores normalizados)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"Índice FAISS construido con {index.ntotal} vectores.")

# Función para embedder de consultas
def embed_query(query):
    return embedding_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)

# Query de ejemplo
query_text = "What is the role of religion in society?"
query_vec = embed_query(query_text)

# Búsqueda top-k
k = 5
start = time.time()
scores, indices = index.search(query_vec, k)
elapsed = time.time() - start
print(f"Tiempo de búsqueda: {elapsed*1000:.2f} ms")

Índice FAISS construido con 3759 vectores.
Tiempo de búsqueda: 0.90 ms


Obtengo los 5 documentos más similares a mi query

In [28]:
print(f"Resultados para la query: '{query_text}'\n")
for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
    print(f"{rank}. Score: {score:.4f}")
    print(f"Categoría: {target_names[targets[idx]]}")
    print(f"Documento: {documents[idx][:300]}...\n")

Resultados para la query: 'What is the role of religion in society?'

1. Score: 0.4589
Categoría: alt.atheism
Documento: [EMAIL] (Tan Chade Meng - dan) writes: Well now, we can't judge death until we are dead right? So, why should we judge religion without having experienced it? People have said that religion is bad by any account, and that it is in no way useful, etc., but I don't totally agree with this. Of course, ...

2. Score: 0.4569
Categoría: soc.religion.christian
Documento: [EMAIL] (edgar pearlstein) writes: : : . : It's my understanding that the U.S. Supreme Court has never : given a legal definition of religion. This despite the many : cases involving religion that have come before the Court. : Can anyone verify or falsify this? : Has any state or other government tr...

3. Score: 0.4372
Categoría: alt.atheism
Documento: [EMAIL] (Sol Lightman) writes: no its not. its due to the fact that there are two issues here: Religion and religion. religion is personal belief system. Re

In [29]:
# Recuperar el texto del documento más relevante
best_doc = documents[indices[0][0]]
prompt = f"Resume el siguiente documento en 3 frases:\n\n{best_doc[:2000]}"

# Usar un modelo disponible en la lista (elige uno de los que aparecen)
response = client.models.generate_content(
    model="gemini-3.1-flash-lite",   # Cambia por gemini-2.5-flash si prefieres
    contents=prompt
)
print("Resumen generado por Gemini:\n", response.text)

Resumen generado por Gemini:
 Aquí tienes el resumen en tres frases:

El autor argumenta que es imposible juzgar la utilidad de la religión sin haberla experimentado personalmente, comparándola con la imposibilidad de evaluar la muerte antes de morir. Reconoce que, aunque no se puede medir el comportamiento de las personas sin la influencia religiosa, es evidente que a algunos individuos la religión les resulta beneficiosa. En conclusión, sostiene que si bien la religión puede no ser el camino adecuado para uno mismo, no se puede descartar su valor para otros.
